In [ ]:
# Додавання бібліотек
import math
import pprint
import random
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd

# Можливі значення змінних
tss_range = (17, 27)
ta_range = (6, 17)
ph_range = (2.8, 4.0)
grape_kinds = ["blue", "green"]

# Кількість екземплярів
samples = 14
generated_data = {
    # Вид винограду
    "kind": [random.choice(grape_kinds) for _ in range(samples)],
    # Точна кислотність
    "ph": [round(random.uniform(*ph_range), 1) for _ in range(samples)],
    # Відносна кислотність
    "ta": [random.randint(*ta_range) for _ in range(samples)],
    # Загальна кількість розчинних твердих речовин
    "tss": [random.randint(*tss_range) for _ in range(samples)],
}
kind_labels = {value: number for number, value in enumerate(grape_kinds)}
print(kind_labels)


{'blue': 0, 'green': 1}


In [79]:
df = pd.DataFrame(data=generated_data)
print(df)

     kind   ph  ta  tss
0    blue  3.8  11   25
1    blue  3.3  17   20
2   green  3.7  14   21
3    blue  3.1  16   25
4    blue  3.8  15   19
5   green  3.2  11   23
6    blue  3.0  16   20
7   green  3.2   8   17
8   green  3.8  11   19
9    blue  3.0   7   27
10   blue  3.5  11   18
11   blue  4.0   6   22
12  green  3.4   6   19
13  green  3.3   8   17


Евклідова норма:

$$
\displaystyle
sum=\sqrt{\sum_{n=1}^{N}{feature_n^2}}
$$

- $N$: кількість екземплярів
- $feature$: ознака, проходить по кожній

$$
\displaystyle
new=\frac{old}{sum}
$$

- $new$: нормалізований екземпляр
- $old$: оригінальний екземпляр

In [80]:
for sample_number, sample_value in df.iterrows():
    df.at[sample_number, "kind"] = kind_labels[df["kind"][sample_number]]

feature_sums = dict()
for feature_name, feature_samples in df.items():
    if feature_name == "kind":
        continue

    df[feature_name] = df[feature_name].astype(float)
    samples_sum = math.sqrt(sum(sample**2 for sample in feature_samples))
    for sample_number, sample_value in enumerate(feature_samples):
        normalized_sample = sample_value / samples_sum
        df.at[sample_number, feature_name] = round(normalized_sample, 2)

    feature_sums[feature_name] = samples_sum
print(df)

   kind    ph    ta   tss
0     0  0.29  0.25  0.32
1     0  0.26  0.38  0.25
2     1  0.29  0.32  0.27
3     0  0.24  0.36  0.32
4     0  0.29  0.34  0.24
5     1  0.25  0.25  0.29
6     0  0.23  0.36  0.25
7     1  0.25  0.18  0.22
8     1  0.29  0.25  0.24
9     0  0.23  0.16  0.34
10    0  0.27  0.25  0.23
11    0  0.31  0.14  0.28
12    1  0.26  0.14  0.24
13    1  0.26  0.18  0.22


Середнє значення:

$$
center=\frac{1}{N}\sum_{n=1}^{N}feature_n
$$

In [81]:
class_centers = dict()
for class_number in kind_labels.values():
    center = {
        "ph": float(round(df[df["kind"] == class_number]["ph"].mean(), 2)),
        "ta": float(round(df[df["kind"] == class_number]["ta"].mean(), 2)),
        "tss": float(round(df[df["kind"] == class_number]["tss"].mean(), 2)),
    }
    class_centers[class_number] = center
pprint.pprint(class_centers)

{0: {'ph': 0.26, 'ta': 0.28, 'tss': 0.28},
 1: {'ph': 0.27, 'ta': 0.22, 'tss': 0.25}}


In [82]:
new_sample = {
    "ph": round(random.uniform(*ph_range), 1),
    "ta": random.randint(*ta_range),
    "tss": random.randint(*tss_range),
}
print(new_sample)

{'ph': 3.9, 'ta': 14, 'tss': 19}


$$
new=\frac{old}{sum}
$$

In [83]:
for feature_name, feature_value in new_sample.items():
    feature_sum = feature_sums[feature_name]
    new_sample[feature_name] = round(feature_value / feature_sum, 2)
print(new_sample)

{'ph': 0.3, 'ta': 0.32, 'tss': 0.24}


$$
distance=\sqrt{\sum_{j=1}^{J}{new_j^2-center_j^2}}
$$

- $J$: кількість ознак

In [84]:
distances = dict()
for class_number in class_centers.keys():
    center_values = np.array(list(class_centers[class_number].values()))
    new_sample_values = np.array(list(new_sample.values()))
    distance_to_class = math.sqrt(sum((new_sample_values - center_values) ** 2))
    distances[distance_to_class] = class_number
print(distances)

{0.06928203230275509: 0, 0.10488088481701516: 1}


In [85]:
sorted_distances = sorted(list(distances.keys()))
print(sorted_distances)

[0.06928203230275509, 0.10488088481701516]


In [86]:
predicted_class = 0
if len(sorted_distances) == 1 or sorted_distances[0] != sorted_distances[1]:
    predicted_class = distances[sorted_distances[0]]
elif sorted_distances[0] == sorted_distances[1]:
    number_of_samples_one = len(df[df["kind"] == distances[sorted_distances[0]]])
    number_of_samples_two = len(df[df["kind"] == distances[sorted_distances[1]]])
    predicted_class = (
        distances[sorted_distances[0]]
        if number_of_samples_one >= number_of_samples_two
        else distances[sorted_distances[1]]
    )
print(predicted_class)

0


In [ ]:
df.loc[-1] = [predicted_class] + list(new_sample.values())
df.index = df.index + 1
df = df.sort_index()
df["kind"] = df["kind"].astype(int)
print(df)

    kind    ph    ta   tss
0      0  0.30  0.32  0.24
1      0  0.29  0.25  0.32
2      0  0.26  0.38  0.25
3      1  0.29  0.32  0.27
4      0  0.24  0.36  0.32
5      0  0.29  0.34  0.24
6      1  0.25  0.25  0.29
7      0  0.23  0.36  0.25
8      1  0.25  0.18  0.22
9      1  0.29  0.25  0.24
10     0  0.23  0.16  0.34
11     0  0.27  0.25  0.23
12     0  0.31  0.14  0.28
13     1  0.26  0.14  0.24
14     1  0.26  0.18  0.22


: 